# Reconstruction du jeu de données — Traducteur Français · Anglais · Éwé

Groupe 03

Ce notebook est celui qui est utilisé pour la construction du corpus d'entraînement.

## Stratégie : la qualité avant le volume

Les évaluations précedentes sur nos données sur FLORES-200 ont montré que la performance du modèle est limitée par la **qualité**
et le **domaine** des données éwé, et non par la taille du modèle. La reconstruction suit donc quatre principes :

1. **Ajouter des sources éwé propres**, traduites par des humains : Bible, *google/SMOL*, *MAFAND*, *FLORES dev*.
2. **Nettoyer agressivement le texte** : normalisation Unicode, suppression du balisage HTML, des URLs et des
   caractères de contrôle, puis **déduplication renforcée** (exacte et normalisée) et **rejet des paires bruitées**
   (copies `source = cible`, lignes essentiellement numériques ou symboliques, ratios de longueur aberrants).
3. **Durcir le filtrage** du corpus web-mined bruité (seuil de similarité cosinus porté à **0,7**).
4. **Réserver FLORES `devtest` à l'évaluation** : seul le découpage `dev` entre dans l'entraînement. Le découpage
   `train` / `valid` / `test` est en outre rendu **insensible au sens de traduction**, afin qu'une paire et son
   inverse tombent toujours dans le même ensemble — aucune fuite possible.

## Étapes

| Étape | Contenu |
|------|---------|
| 0 | Configuration, nettoyage du texte et fonctions outils |
| 1 | Collecte des sources (OPUS, Ghana, SMOL, FLORES, MAFAND) |
| 2 | Alignement par similarité cosinus |
| 3 | Filtrage par identification de langue |
| 4 | Filtrage par caractères spécifiques à l'éwé |
| 5 | Durcissement du seuil cosinus à 0,7 |
| 6 | Filtres anti-bruit, déduplication et construction des ensembles `train` / `valid` / `test` |
| 7 | Bilan du jeu de données final |

## 0. Configuration, nettoyage du texte et fonctions outils

On se place à la racine du projet, on définit les dossiers de travail, les correspondances de codes de langue
(éwé, français, anglais), ainsi que les constantes du pipeline :

- `ALIGN_THRESHOLD` : seuil cosinus de base appliqué lors de l'alignement (étape 2).
- `MIN_COS` : seuil durci appliqué au web-mined avant la construction finale (étape 5).

Le cœur de cette étape est `clean_txt`, le **nettoyeur de texte unique** appliqué à chaque chaîne du corpus.
Il **normalise l'Unicode en NFC** (indispensable pour que les diacritiques éwé soient comparables), **supprime
le balisage HTML, les URLs, les adresses e-mail et les caractères de contrôle** souvent laissés par le web-mining,
puis réduit toutes les espaces (y compris insécables) à une seule. Une chaîne devenue vide après nettoyage entraîne
le rejet de la paire.

Les autres fonctions assurent la canonisation des langues, le calcul d'un identifiant stable et l'écriture d'un CSV
au format canonique `id, src_lang, tgt_lang, src_text, tgt_text, source`, avec **déduplication exacte** intégrée.

In [1]:
import os, re, glob, csv, time, hashlib, unicodedata, urllib.request, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from tqdm import tqdm
tqdm.pandas()

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
print('Racine du projet :', Path.cwd())

OPUS = Path('data/processed/opus')
HF = Path('data/processed/hf')
CLEAN = Path('data/processed/clean')
NLLB = Path('data/processed/nllb')
RAW = Path('data/raw')
for d in (OPUS, HF, CLEAN, NLLB, RAW):
    d.mkdir(parents=True, exist_ok=True)

MODEL = 'facebook/nllb-200-distilled-600M'
DEVICE = 'cuda'
ALIGN_THRESHOLD = 0.5
MIN_COS = 0.7

NLLB_CODE = {'fra': 'fra_Latn', 'eng': 'eng_Latn', 'ewe': 'ewe_Latn'}
ALIAS = {'fr': 'fra', 'fra': 'fra', 'french': 'fra', 'en': 'eng', 'eng': 'eng',
         'english': 'eng', 'ee': 'ewe', 'ewe': 'ewe'}

canon_lang = lambda v: ALIAS[str(v).strip().lower()]

_TAG = re.compile(r'<[^>]+>')
_URL = re.compile(r'https?://\S+|www\.\S+')
_EMAIL = re.compile(r'\S+@\S+\.\S+')
_CTRL = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')

def clean_txt(s):
    s = unicodedata.normalize('NFC', str(s))
    s = _TAG.sub(' ', s)
    s = _URL.sub(' ', s)
    s = _EMAIL.sub(' ', s)
    s = _CTRL.sub(' ', s)
    s = s.replace('\xa0', ' ').replace('​', '').replace('‎', '').replace('﻿', '')
    return ' '.join(s.split())

def sid(*parts):
    return hashlib.sha1('|'.join(map(str, parts)).encode()).hexdigest()

def write_canon(path, rows):
    seen, n = set(), 0
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['id', 'src_lang', 'tgt_lang', 'src_text', 'tgt_text', 'source'])
        for sl, tl, st, tt, src in rows:
            st, tt = clean_txt(st), clean_txt(tt)
            if not st or not tt:
                continue
            key = (sl, tl, st, tt)
            if key in seen:
                continue
            seen.add(key)
            w.writerow([sid(src, sl, tl, st, tt), sl, tl, st, tt, src])
            n += 1
    print(f'{path} : {n} paires')
    return n

Racine du projet : D:\python\traducteur-fr-en-ewe


## 1. Collecte des sources

### 1a. Corpus OPUS

On télécharge les corpus parallèles d'OPUS et on les convertit au format canonique :

- **Français ↔ anglais propres** : TED2020 (exposés), Tatoeba (phrases courtes), News-Commentary (actualité).
- **Éwé web-mined** : OPUS-NLLB éwé↔français **et** éwé↔anglais (volumineux mais **bruités**, à filtrer fortement),
  et translatewiki. La paire éwé↔anglais renforce la direction `en↔ewe`, la plus faible du modèle.
- **Bible** (`bible-uedin`) : éwé ↔ anglais et éwé ↔ français, **propre et bien aligné** mais de domaine religieux
  (il sera donc plafonné lors de la construction finale).

Les archives OPUS sont au format *Moses* (un fichier texte par langue, lignes alignées). Un téléchargement déjà
présent n'est pas refait.

In [2]:
OPUS_SOURCES = {
    'nllb_ee_fr': ('https://object.pouta.csc.fi/OPUS-NLLB/v1/moses/ee-fr.txt.zip', 'ewe', 'fra', 'ee', 'fr'),
    'nllb_ee_en': ('https://object.pouta.csc.fi/OPUS-NLLB/v1/moses/en-ee.txt.zip', 'ewe', 'eng', 'ee', 'en'),
    'translatewiki_ee_fr': ('https://object.pouta.csc.fi/OPUS-translatewiki/v2025-01-01/moses/ee-fr.txt.zip', 'ewe', 'fra', 'ee', 'fr'),
    'ted2020_en_fr': ('https://object.pouta.csc.fi/OPUS-TED2020/v1/moses/en-fr.txt.zip', 'eng', 'fra', 'en', 'fr'),
    'news_commentary_en_fr': ('https://object.pouta.csc.fi/OPUS-News-Commentary/v16/moses/en-fr.txt.zip', 'eng', 'fra', 'en', 'fr'),
    'tatoeba_en_fr': ('https://object.pouta.csc.fi/OPUS-Tatoeba/v2023-04-12/moses/en-fr.txt.zip', 'eng', 'fra', 'en', 'fr'),
    'bible_ee_en': ('https://object.pouta.csc.fi/OPUS-bible-uedin/v1/moses/ee-en.txt.zip', 'ewe', 'eng', 'ee', 'en'),
    'bible_ee_fr': ('https://object.pouta.csc.fi/OPUS-bible-uedin/v1/moses/ee-fr.txt.zip', 'ewe', 'fra', 'ee', 'fr'),
}

def opus_to_csv(key):
    out = OPUS / f'{key}.csv'
    if out.exists():
        print(f'{out.name} déjà présent')
        return
    url, sl, tl, code_src, code_tgt = OPUS_SOURCES[key]
    zip_path = RAW / f'{key}.zip'
    if not zip_path.exists():
        print(f'Téléchargement de {key}')
        urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        pick = lambda code: sorted([n for n in names if n.endswith('.' + code)], key=len)[0]
        src = [l.decode('utf-8', 'replace').strip() for l in z.open(pick(code_src))]
        tgt = [l.decode('utf-8', 'replace').strip() for l in z.open(pick(code_tgt))]
    write_canon(out, [(sl, tl, a, b, f'opus:{key}') for a, b in zip(src, tgt)])

for key in OPUS_SOURCES:
    opus_to_csv(key)

nllb_ee_fr.csv déjà présent
nllb_ee_en.csv déjà présent
translatewiki_ee_fr.csv déjà présent
ted2020_en_fr.csv déjà présent
news_commentary_en_fr.csv déjà présent
tatoeba_en_fr.csv déjà présent
bible_ee_en.csv déjà présent
bible_ee_fr.csv déjà présent


### 1b. Corpus Ghana NLP (anglais ↔ éwé)

Le corpus `english-ewe-sentence-pairs-4m` est le plus volumineux pour cette paire, mais il s'est révélé
**massivement mésaligné** (de l'ordre de 95 % de paires incohérentes). On l'importe en *streaming* en ne
gardant que les lignes dont le score de similarité fourni est ≥ 0,80, et on plafonne à 120 000 lignes.
Il sera de toute façon fortement filtré par les étapes suivantes.

In [3]:
ghana = HF / 'ghana_en_ewe.csv'
if ghana.exists():
    print(f'{ghana} déjà importé')
else:
    from datasets import load_dataset
    ds = load_dataset('ghananlpcommunity/english-ewe-sentence-pairs-4m', split='train', streaming=True)
    rows = []
    for r in ds:
        try:
            quality = float(r.get('similarity', 0) or 0)
        except (TypeError, ValueError):
            quality = 0
        if quality < 0.80:
            continue
        st, tt = clean_txt(r.get('English')), clean_txt(r.get('Ewe'))
        if st and tt:
            rows.append(('eng', 'ewe', st, tt, 'hf:ghana'))
        if len(rows) >= 200000:
            break
    write_canon(ghana, rows)

data\processed\hf\ghana_en_ewe.csv déjà importé


### 1c. Sources éwé propres (traduites par des humains)

Ces trois sources sont de **haute qualité** et n'ont pas besoin du filtrage anti-bruit :

- **google/SMOL** (`en-ee`) : traductions professionnelles de phrases et de documents, domaine général.
- **FLORES dev** : phrases alignées en éwé, français et anglais. On n'utilise que le découpage `dev` ;
  `devtest` est conservé pour l'évaluation finale.
- **MAFAND** (`fr-ewe`) : corpus d'actualité du projet Masakhane, traduit par des humains.

In [4]:
from datasets import load_dataset

rows = []
ds = load_dataset('google/smol', 'smolsent__en_ee', split='train')
rows += [('eng', 'ewe', r['src'], r['trg'], 'smol:smolsent') for r in ds]
ds = load_dataset('google/smol', 'smoldoc__en_ee', split='train')
for r in ds:
    rows += [('eng', 'ewe', s, t, 'smol:smoldoc') for s, t in zip(r['srcs'], r['trgs'])]
write_canon(CLEAN / 'smol_en_ewe.csv', rows)

D:\python\traducteur-fr-en-ewe\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data\processed\clean\smol_en_ewe.csv : 3230 paires


3230

In [5]:
ds = load_dataset('facebook/flores', 'all', split='dev', trust_remote_code=True)
rows = []
for r in ds:
    en, fr, ewe = r['sentence_eng_Latn'], r['sentence_fra_Latn'], r['sentence_ewe_Latn']
    rows += [('eng', 'ewe', en, ewe, 'flores:dev'),
             ('fra', 'ewe', fr, ewe, 'flores:dev'),
             ('eng', 'fra', en, fr, 'flores:dev')]
write_canon(CLEAN / 'flores_dev.csv', rows)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'facebook/flores' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


data\processed\clean\flores_dev.csv : 2991 paires


2991

In [6]:
base = 'https://raw.githubusercontent.com/masakhane-io/lafand-mt/main/data/text_files/fr-ewe/'
rows = []
for split in ['train', 'dev']:
    try:
        fr = urllib.request.urlopen(base + f'{split}.fr').read().decode().splitlines()
        ewe = urllib.request.urlopen(base + f'{split}.ewe').read().decode().splitlines()
        if len(fr) == len(ewe):
            rows += [('fra', 'ewe', a, b, 'mafand:fr-ewe') for a, b in zip(fr, ewe)]
    except Exception as exc:
        print(split, 'erreur', str(exc)[:100])
write_canon(CLEAN / 'mafand_fr_ewe.csv', rows)

data\processed\clean\mafand_fr_ewe.csv : 3438 paires


3438

## 2. Alignement par similarité cosinus

Pour chaque paire, on encode les deux côtés avec l'**encodeur de NLLB-200** (moyenne masquée des états cachés,
puis normalisation L2) et on calcule la **similarité cosinus**. Les paires dont les deux côtés ne se correspondent
pas obtiennent un cosinus faible et sont écartées (seuil de base 0,5). Le score est conservé dans la colonne
`quality_score`.

Cette étape s'exécute sur **GPU** et constitue la partie la plus lourde. Si les fichiers `*_aligned.csv` existent
déjà, elle est **réutilisée** sans recalcul. Seul le corpus web-mined y passe ; les sources propres en sont exemptées.

In [7]:
EMB_BATCH = 256

def cosine_align(files, threshold):
    import torch
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    if all(Path(p).with_name(Path(p).stem + '_aligned.csv').exists() for p in files):
        print('Alignement déjà effectué, réutilisation des fichiers existants')
        return
    print(f'Chargement de l\'encodeur {MODEL} ...')
    tok = AutoTokenizer.from_pretrained(MODEL)
    enc = AutoModelForSeq2SeqLM.from_pretrained(MODEL).get_encoder().to(DEVICE).eval()
    bf16 = DEVICE == 'cuda' and torch.cuda.is_bf16_supported()
    print(f'Périphérique : {DEVICE} | bf16 : {bf16} | batch : {EMB_BATCH}')

    @torch.no_grad()
    def embed(texts, lang):
        tok.src_lang = NLLB_CODE[lang]
        out = []
        ctx = torch.autocast('cuda', dtype=torch.bfloat16) if bf16 else torch.autocast('cpu', enabled=False)
        for i in tqdm(range(0, len(texts), EMB_BATCH), desc=f'      encodage {lang}', unit='lot', leave=False):
            batch = [str(x) for x in texts[i:i + EMB_BATCH]]
            enc_in = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=128).to(DEVICE)
            with ctx:
                hidden = enc(**enc_in).last_hidden_state
            mask = enc_in['attention_mask'].unsqueeze(-1).float()
            pooled = (hidden.float() * mask).sum(1) / mask.sum(1).clamp(min=1)
            out.append(torch.nn.functional.normalize(pooled, dim=-1).cpu().numpy())
        return np.concatenate(out)

    def embed_df(df, text_col, lang_col):
        vecs = np.zeros((len(df), enc.config.d_model), dtype=np.float32)
        for lang, idx in df.groupby(lang_col).groups.items():
            idx = list(idx)
            vecs[[df.index.get_loc(i) for i in idx]] = embed(df.loc[idx, text_col].tolist(), canon_lang(lang))
        return vecs

    for n, p in enumerate(files, 1):
        out = Path(p).with_name(Path(p).stem + '_aligned.csv')
        if out.exists():
            print(f'[{n}/{len(files)}] {out.name} déjà présent')
            continue
        df = pd.read_csv(p).reset_index(drop=True)
        
        MAX_PAIRS = 300_000
        if len(df) > MAX_PAIRS:
            print(f'      Limitation à {MAX_PAIRS:,} paires')
            df = df.sample(MAX_PAIRS, random_state=42).reset_index(drop=True)
            
        print(f'[{n}/{len(files)}] {Path(p).name} : encodage de {len(df)} paires ({2 * len(df)} phrases)...')
        t0 = time.time()
        cos = (embed_df(df, 'src_text', 'src_lang') * embed_df(df, 'tgt_text', 'tgt_lang')).sum(1)
        df['quality_score'] = cos
        kept = df[cos >= threshold]
        kept.to_csv(out, index=False)
        print(f'      {len(df)} -> {len(kept)} gardées (cosinus médian {np.median(cos):.2f}, {time.time() - t0:.0f} s)')

noisy = ['data/processed/opus/nllb_ee_fr.csv', 'data/processed/opus/nllb_ee_en.csv', 'data/processed/hf/ghana_en_ewe.csv',
         'data/processed/opus/ted2020_en_fr.csv', 'data/processed/opus/tatoeba_en_fr.csv',
         'data/processed/opus/news_commentary_en_fr.csv', 'data/processed/opus/translatewiki_ee_fr.csv']
cosine_align(noisy, ALIGN_THRESHOLD)

Chargement de l'encodeur facebook/nllb-200-distilled-600M ...


Loading weights: 100%|██████████| 512/512 [00:00<00:00, 19185.27it/s]


Périphérique : cuda | bf16 : True | batch : 256
[1/7] nllb_ee_fr_aligned.csv déjà présent
      Limitation à 300,000 paires
[2/7] nllb_ee_en.csv : encodage de 300000 paires (600000 phrases)...


      300000 -> 251281 gardées (cosinus médian 0.65, 344 s)
[3/7] ghana_en_ewe.csv : encodage de 120000 paires (240000 phrases)...


      120000 -> 6099 gardées (cosinus médian 0.34, 123 s)
      Limitation à 300,000 paires
[4/7] ted2020_en_fr.csv : encodage de 300000 paires (600000 phrases)...


      300000 -> 298779 gardées (cosinus médian 0.87, 530 s)
[5/7] tatoeba_en_fr.csv : encodage de 274883 paires (549766 phrases)...


      274883 -> 274388 gardées (cosinus médian 0.89, 199 s)
[6/7] news_commentary_en_fr.csv : encodage de 151421 paires (302842 phrases)...


      151421 -> 149487 gardées (cosinus médian 0.86, 257 s)
[7/7] translatewiki_ee_fr.csv : encodage de 1270 paires (2540 phrases)...


      1270 -> 948 gardées (cosinus médian 0.65, 2 s)


## 3. Filtrage par identification de langue

L'alignement cosinus ne détecte pas la **contamination de langue** : une paire bien traduite mais dont la cible
est écrite dans la mauvaise langue obtient un bon cosinus. On vérifie donc, avec le modèle officiel
`facebook/fasttext-language-identification`, que chaque côté est réellement écrit dans la langue déclarée
(confiance minimale 0,5). Réutilisé si déjà effectué.

In [8]:
def langid_filter(files, min_prob=0.5):
    if all(Path(p).with_name(Path(p).stem + '_lid.csv').exists() for p in files):
        print('Filtrage de langue déjà effectué, réutilisation des fichiers existants')
        return
    import fasttext
    from huggingface_hub import hf_hub_download
    print('Chargement du modèle fastText d\'identification de langue ...')
    model = fasttext.load_model(hf_hub_download('facebook/fasttext-language-identification', 'model.bin'))

    def predict(texts):
        cleaned = [t.replace('\n', ' ').replace('\r', ' ').strip() for t in texts]
        labels, probs = model.predict(cleaned, k=1)
        return ([l[0].replace('__label__', '') if l else '' for l in labels],
                [float(p[0]) if len(p) else 0.0 for p in probs])

    for n, p in enumerate(files, 1):
        out = Path(p).with_name(Path(p).stem + '_lid.csv')
        if out.exists():
            print(f'[{n}/{len(files)}] {out.name} déjà présent')
            continue
        df = pd.read_csv(p).reset_index(drop=True)
        print(f'[{n}/{len(files)}] {Path(p).name} : identification de langue sur {len(df)} paires...')
        df['src_text'] = df['src_text'].fillna('').astype(str)
        df['tgt_text'] = df['tgt_text'].fillna('').astype(str)
        exp_src = df['src_lang'].map(lambda v: NLLB_CODE[canon_lang(v)])
        exp_tgt = df['tgt_lang'].map(lambda v: NLLB_CODE[canon_lang(v)])
        pred_src, prob_src = predict(df['src_text'].tolist())
        pred_tgt, prob_tgt = predict(df['tgt_text'].tolist())
        ok = ((pd.Series(pred_src, index=df.index) == exp_src) & (pd.Series(prob_src, index=df.index) >= min_prob)) & \
             ((pd.Series(pred_tgt, index=df.index) == exp_tgt) & (pd.Series(prob_tgt, index=df.index) >= min_prob))
        df[ok].to_csv(out, index=False)
        print(f'      {len(df)} -> {int(ok.sum())}')

langid_filter(glob.glob('data/processed/opus/*_aligned.csv') + glob.glob('data/processed/hf/*_aligned.csv'))

Chargement du modèle fastText d'identification de langue ...
[1/6] news_commentary_en_fr_aligned.csv : identification de langue sur 149487 paires...
      149487 -> 148253
[2/6] nllb_ee_en_aligned.csv : identification de langue sur 251281 paires...
      251281 -> 251115
[3/6] nllb_ee_fr_aligned.csv : identification de langue sur 816430 paires...
      816430 -> 815866
[4/6] tatoeba_en_fr_aligned.csv : identification de langue sur 274388 paires...
      274388 -> 263411
[5/6] ted2020_en_fr_aligned.csv : identification de langue sur 298779 paires...
      298779 -> 292030
[6/6] translatewiki_ee_fr_aligned.csv : identification de langue sur 948 paires...
      948 -> 617


## 4. Filtrage par caractères spécifiques à l'éwé

fastText ne sépare pas l'anglais de l'éwé sur les textes courts : il étiquette `ewe_Latn` avec une forte confiance
même pour de l'anglais évident. Le seul signal fiable est **orthographique** : l'éwé emploie des lettres qui
n'existent pas en anglais (`ɖ ƒ ŋ ɔ ɛ ɣ ʋ`, leurs majuscules, voyelles nasalisées et diacritiques de ton).
On exige donc qu'un côté déclaré éwé contienne au moins un de ces caractères. Le compromis assumé est de supprimer
aussi le vrai éwé écrit en pur ASCII, au profit d'une précision maximale.

In [9]:
EWE_CHARS = set('ɖƒŋɔɛɣʋƉƑŊƆƐƔƲãẽĩõũ' + '̣̃̀́̄̌')
has_ewe = lambda t: any(c in EWE_CHARS for c in str(t))

def ewe_filter(files):
    for n, p in enumerate(files, 1):
        out = Path(p).with_name(Path(p).stem + '_ewe.csv')
        if out.exists():
            print(f'[{n}/{len(files)}] {out.name} déjà présent')
            continue
        df = pd.read_csv(p).reset_index(drop=True)
        df['src_text'] = df['src_text'].fillna('').astype(str)
        df['tgt_text'] = df['tgt_text'].fillna('').astype(str)
        src_ewe = df['src_lang'].map(lambda v: canon_lang(v) == 'ewe')
        tgt_ewe = df['tgt_lang'].map(lambda v: canon_lang(v) == 'ewe')
        bad = (src_ewe & ~df['src_text'].map(has_ewe)) | (tgt_ewe & ~df['tgt_text'].map(has_ewe))
        df[~bad].to_csv(out, index=False)
        print(f'[{n}/{len(files)}] {Path(p).name} : {len(df)} -> {int((~bad).sum())}')

ewe_filter(glob.glob('data/processed/opus/*_lid.csv') + glob.glob('data/processed/hf/*_lid.csv'))

[1/6] news_commentary_en_fr_aligned_lid.csv : 148253 -> 148253
[2/6] nllb_ee_en_aligned_lid.csv : 251115 -> 160862
[3/6] nllb_ee_fr_aligned_lid.csv : 815866 -> 491081
[4/6] tatoeba_en_fr_aligned_lid.csv : 263411 -> 263411
[5/6] ted2020_en_fr_aligned_lid.csv : 292030 -> 292030
[6/6] translatewiki_ee_fr_aligned_lid.csv : 617 -> 573


## 5. Durcissement du seuil cosinus à 0,7

Le cosinus de chaque paire est déjà calculé et stocké (étape 2). On peut donc resserrer le seuil à **0,7**
de façon **instantanée**, sans recalcul, pour ne garder que les paires web-mined les plus fiables.

In [10]:
q07 = []
for p in glob.glob('data/processed/opus/*_aligned_lid_ewe.csv') + glob.glob('data/processed/hf/*_aligned_lid_ewe.csv'):
    df = pd.read_csv(p)
    if 'quality_score' in df.columns:
        before = len(df)
        df = df[pd.to_numeric(df['quality_score'], errors='coerce').fillna(0) >= MIN_COS]
        print(f'{Path(p).name} : {before} -> {len(df)}')
    out = p.replace('_lid_ewe.csv', f'_q{int(MIN_COS * 100)}.csv')
    df.to_csv(out, index=False)
    q07.append(out)
print('Fichiers durcis :', len(q07))

news_commentary_en_fr_aligned_lid_ewe.csv : 148253 -> 134678
nllb_ee_en_aligned_lid_ewe.csv : 160862 -> 91779
nllb_ee_fr_aligned_lid_ewe.csv : 491081 -> 288617
tatoeba_en_fr_aligned_lid_ewe.csv : 263411 -> 258547
ted2020_en_fr_aligned_lid_ewe.csv : 292030 -> 283008
translatewiki_ee_fr_aligned_lid_ewe.csv : 573 -> 286
Fichiers durcis : 6


## 6. Filtres anti-bruit, déduplication et construction des ensembles

On assemble le web-mined durci, la Bible et les sources propres, puis la fonction de construction applique
successivement une chaîne de garde-fous qualité :

1. **Seuil de qualité** : on ne garde que les paires de cosinus ≥ `min_quality` (les sources propres valent 1,0).
2. **Filtres anti-bruit** :
   - **rejet des copies** où la source est identique à la cible (après normalisation casse/ponctuation) — typique
     des lignes non traduites du web-mining ;
   - **rejet des lignes pauvres en lettres** (essentiellement chiffres, codes ou symboles) via un ratio
     alphabétique minimal ;
   - **filtre de longueur** en caractères et **ratio de longueur** borné entre les deux côtés ;
   - **ratio du nombre de mots** borné, pour écarter les alignements grossièrement déséquilibrés.
3. **Déduplication renforcée** : déduplication exacte, puis **normalisée** (insensible à la casse et à la
   ponctuation) afin d'éliminer les quasi-doublons résiduels entre sources.
4. **Plafond du domaine religieux** (au plus 20 %) pour limiter le biais biblique de l'éwé.
5. **Bidirectionnalisation** : chaque paire engendre les deux sens, puis un **plafond par direction**.
6. **Découpage déterministe et insensible au sens** : le bucket de découpage est calculé sur une **clé canonique**
   (langues et textes triés). Une paire et son inverse obtiennent donc le **même** bucket et tombent dans le même
   ensemble — il est impossible qu'un exemple fuite de l'entraînement vers le test via sa traduction inverse.

In [11]:
RELIGIOUS = ('bible', 'jw300', 'tanzil', 'quran')
is_relig = lambda s: any(m in str(s).lower() for m in RELIGIOUS)
norm_key = lambda s: re.sub(r'[^\w\s]', '', str(s).lower())
alpha_ratio = lambda s: sum(c.isalpha() for c in str(s)) / max(len(str(s)), 1)

def load_canon(p):
    df = pd.read_csv(p)
    df['src_lang'] = df['src_lang'].map(canon_lang)
    df['tgt_lang'] = df['tgt_lang'].map(canon_lang)
    df['src_text'] = df['src_text'].fillna('').map(clean_txt)
    df['tgt_text'] = df['tgt_text'].fillna('').map(clean_txt)
    df['source'] = df['source'].fillna(Path(p).stem).astype(str)
    if 'quality_score' not in df.columns:
        df['quality_score'] = 1.0
    df['is_religious'] = df['source'].map(is_relig)
    return df[['src_lang', 'tgt_lang', 'src_text', 'tgt_text', 'source', 'quality_score', 'is_religious']]

def build(inputs, min_quality=0.7, max_relig=0.2, max_per_dir=120000,
          min_chars=2, max_chars=512, max_char_ratio=3.0, max_word_ratio=4.0,
          min_alpha=0.4, valid_ratio=0.02, test_ratio=0.02, seed=42):
    df = pd.concat([load_canon(p) for p in tqdm(inputs, desc='Chargement des sources', unit='fichier')], ignore_index=True)
    print(f'  chargé : {len(df)} paires')
    df = df[pd.to_numeric(df['quality_score'], errors='coerce').fillna(1.0) >= min_quality]
    df = df[(df['src_text'].str.len() > 0) & (df['tgt_text'].str.len() > 0)]
    print(f'  après seuil qualité ≥ {min_quality} : {len(df)}')

    df = df[df['src_text'].map(norm_key) != df['tgt_text'].map(norm_key)]
    src_len, tgt_len = df['src_text'].str.len(), df['tgt_text'].str.len()
    char_ratio = pd.concat([src_len, tgt_len], axis=1).max(axis=1) / pd.concat([src_len, tgt_len], axis=1).min(axis=1).clip(lower=1)
    src_words, tgt_words = df['src_text'].str.split().map(len), df['tgt_text'].str.split().map(len)
    word_ratio = pd.concat([src_words, tgt_words], axis=1).max(axis=1) / pd.concat([src_words, tgt_words], axis=1).min(axis=1).clip(lower=1)
    df = df[(src_len >= min_chars) & (tgt_len >= min_chars) & (src_len <= max_chars) & (tgt_len <= max_chars) &
            (char_ratio <= max_char_ratio) & (word_ratio <= max_word_ratio) &
            (df['src_text'].map(alpha_ratio) >= min_alpha) & (df['tgt_text'].map(alpha_ratio) >= min_alpha)]
    print(f'  après filtres anti-bruit : {len(df)}')

    df = df.drop_duplicates(['src_lang', 'tgt_lang', 'src_text', 'tgt_text'])
    df['_ks'], df['_kt'] = df['src_text'].map(norm_key), df['tgt_text'].map(norm_key)
    df = df.drop_duplicates(['src_lang', 'tgt_lang', '_ks', '_kt']).drop(columns=['_ks', '_kt'])
    print(f'  après déduplication : {len(df)}')

    rev = df.rename(columns={'src_lang': 'tgt_lang', 'tgt_lang': 'src_lang', 'src_text': 'tgt_text', 'tgt_text': 'src_text'})
    df = pd.concat([df, rev], ignore_index=True)
    print(f'  après bidirectionnalisation : {len(df)}')
    non_relig, relig = df[~df['is_religious']], df[df['is_religious']]
    cap = int(len(non_relig) * max_relig)
    if len(relig) > cap:
        relig = relig.sample(cap, random_state=seed)
    df = pd.concat([non_relig, relig], ignore_index=True)
    print(f'  après plafond religieux ({max_relig:.0%}) : {len(df)}')
    parts = []
    for _, group in df.groupby(['src_lang', 'tgt_lang'], sort=False):
        parts.append(group.sample(max_per_dir, random_state=seed) if len(group) > max_per_dir else group)
    df = pd.concat(parts, ignore_index=True)
    print(f'  après plafond par direction ({max_per_dir}) : {len(df)}')

    def split_of(r):
        a, b = sorted([r['src_lang'], r['tgt_lang']])
        s, t = sorted([r['src_text'], r['tgt_text']])
        bucket = int(sid(a, b, s, t)[:8], 16) / 0xFFFFFFFF
        if bucket < test_ratio:
            return 'test'
        return 'valid' if bucket < test_ratio + valid_ratio else 'train'

    tqdm.pandas(desc='Découpage train/valid/test')
    df['split'] = df.progress_apply(split_of, axis=1)
    df['src_lang_nllb'] = df['src_lang'].map(NLLB_CODE)
    df['tgt_lang_nllb'] = df['tgt_lang'].map(NLLB_CODE)
    tqdm.pandas(desc='Calcul des identifiants')
    df['id'] = df.progress_apply(lambda r: sid(r['split'], r['src_lang'], r['tgt_lang'], r['src_text'], r['tgt_text']), axis=1)
    df['domain'] = ''
    df['license'] = ''
    cols = ['id', 'split', 'src_lang', 'tgt_lang', 'src_lang_nllb', 'tgt_lang_nllb',
            'src_text', 'tgt_text', 'source', 'domain', 'quality_score', 'is_religious', 'license']
    df = df[cols].sort_values(['split', 'src_lang', 'tgt_lang', 'source']).reset_index(drop=True)
    print('  écriture des CSV ...')
    df.to_csv(NLLB / 'nllb_all.csv', index=False)
    for split, group in df.groupby('split'):
        group.to_csv(NLLB / f'nllb_{split}.csv', index=False)
    summary = df.groupby(['split', 'src_lang', 'tgt_lang', 'source', 'is_religious']).size().reset_index(name='rows')
    summary.to_csv(NLLB / 'nllb_summary.csv', index=False)
    return df

inputs = list(q07) + ['data/processed/opus/bible_ee_en.csv', 'data/processed/opus/bible_ee_fr.csv'] + glob.glob('data/processed/clean/*.csv')
for i in inputs:
    print('Source :', i)
data = build(inputs, min_quality=MIN_COS)
print('Total lignes :', len(data))

Source : data/processed/opus\news_commentary_en_fr_aligned_q70.csv
Source : data/processed/opus\nllb_ee_en_aligned_q70.csv
Source : data/processed/opus\nllb_ee_fr_aligned_q70.csv
Source : data/processed/opus\tatoeba_en_fr_aligned_q70.csv
Source : data/processed/opus\ted2020_en_fr_aligned_q70.csv
Source : data/processed/opus\translatewiki_ee_fr_aligned_q70.csv
Source : data/processed/opus/bible_ee_en.csv
Source : data/processed/opus/bible_ee_fr.csv
Source : data/processed/clean\flores_dev.csv
Source : data/processed/clean\mafand_fr_ewe.csv
Source : data/processed/clean\smol_en_ewe.csv


Chargement des sources: 100%|██████████| 11/11 [00:16<00:00,  1.47s/fichier]


  chargé : 1090357 paires
  après seuil qualité ≥ 0.7 : 1090357
  après filtres anti-bruit : 1088846
  après déduplication : 1064356
  après bidirectionnalisation : 2128712
  après plafond religieux (20%) : 2128712
  après plafond par direction (120000) : 697744


Calcul des identifiants: 100%|██████████| 697744/697744 [00:11<00:00, 62704.00it/s]


  écriture des CSV ...
Total lignes : 697744


## 7. Bilan du jeu de données final

Tailles des ensembles, contrôle d'**absence de fuite** entre `train` et `test` (sur la clé canonique
insensible au sens), puis répartition des paires d'entraînement par direction et par source.

In [12]:
for split in ['train', 'valid', 'test']:
    print(split, ':', len(pd.read_csv(NLLB / f'nllb_{split}.csv')), 'lignes')

train = pd.read_csv(NLLB / 'nllb_train.csv')
test = pd.read_csv(NLLB / 'nllb_test.csv')
ckey = lambda d: {sid(*sorted([a, b]), *sorted([s, t]))
                  for a, b, s, t in zip(d['src_lang'], d['tgt_lang'], d['src_text'], d['tgt_text'])}
leak = ckey(train) & ckey(test)
print('\nFuite train/test (clé canonique) :', len(leak), 'paires communes')

print('\nPaires par direction :')
print(train.groupby(['src_lang', 'tgt_lang']).size().to_string())
print('\nPaires par direction et par source :')
print(train.groupby(['src_lang', 'tgt_lang', 'source']).size().to_string())

train : 669904 lignes
valid : 13853 lignes
test : 13987 lignes

Fuite train/test (clé canonique) : 0 paires communes

Paires par direction :
src_lang  tgt_lang
eng       ewe         104499
          fra         115223
ewe       eng         104499
          fra         115184
fra       eng         115223
          ewe         115276

Paires par direction et par source :
src_lang  tgt_lang  source                    
eng       ewe       flores:dev                       950
                    opus:bible_ee_en               15063
                    opus:nllb_ee_en                85380
                    smol:smoldoc                    2267
                    smol:smolsent                    839
          fra       flores:dev                       167
                    opus:news_commentary_en_fr     22773
                    opus:tatoeba_en_fr             44080
                    opus:ted2020_en_fr             48203
ewe       eng       flores:dev                       950
           

---

Le jeu de données est prêt. On peut passer à l'entraînement avec le notebook `01_pipeline_nllb_lora.ipynb`
en pointant `TRAIN_CSV` sur `data/processed/nllb/nllb_train.csv`.